# Simulation et métriques de jonglage

Choisir le kernel **Python (so101-robotic-learning)** et exécuter les cellules dans l'ordre.
Les actions ci-dessous sont aléatoires et reproductibles : ce notebook vérifie l'environnement,
pas la qualité d'une politique apprise. Zéro rebond est un résultat possible et normal.
Les exemples utilisent les mêmes bonus de rebond, shaping et pénalité de chute que l'entraînement.
Le backend graphique reste celui de la plateforme ; aucune fenêtre n'est ouverte par défaut.


In [ ]:
from datetime import datetime
from pathlib import Path
import json
import numpy as np
from IPython.display import JSON, Markdown, display

# Works from the repository root or any of its subdirectories.
root = next((path for path in (Path.cwd(), *Path.cwd().parents)
             if (path / "pyproject.toml").is_file()
             and (path / "src/so101_robotic_learning").is_dir()), None)
if root is None:
    raise RuntimeError("Ouvrir ce notebook depuis le dépôt ou l'un de ses sous-dossiers.")
xml_path = root / "assets/mjcf/so101_new_calib copy.xml"
if not xml_path.is_file():
    raise FileNotFoundError(xml_path)

# MuJoCo selects the platform backend. Do not force EGL on Windows.
from so101_robotic_learning.bounce_rl.env.bounce_events import BounceConfig
from so101_robotic_learning.bounce_rl.rewards import (
    RewardMultiObjectif, BounceReward, FailurePenalty, EnergyPenalty, BallSegmentPenalty,
)

import time
import matplotlib.pyplot as plt
from matplotlib import animation
from IPython.display import HTML
from so101_robotic_learning.bounce_rl.env.bounce_env import BounceEnv
from so101_robotic_learning.bounce_rl.rewards.reward_ball_aligned_on_z_and_above_paddle import BallAlignedOnZAndAbovePaddleReward


In [ ]:
env_kwargs = {
    "camera_name": "gripper_cam",  # policy input only
    "video_camera_name": "overview_cam",  # visualization only
    "video_width": 320,
    "video_height": 240,
    "bounce_config": BounceConfig(success_bounces=5),
    "max_episode_steps": 250,
    "frame_skip": 20,
    "camera_fps": 30,
    "width": 320,
    "height": 240,
    "latent_steps": np.array([], dtype=np.int32),
}
reward_objective = (
    RewardMultiObjectif()
    .add_reward("bounces", BounceReward(), weight=5.0)
    .add_reward("shaping", BallAlignedOnZAndAbovePaddleReward(), weight=0.05)
    .add_penalty("failure", FailurePenalty(), weight=5.0)
    .add_penalty("energy", EnergyPenalty(), weight=0.01)
    .add_penalty("ball_segment", BallSegmentPenalty(), weight=1.0)
)

SIM_DURATION = 5.0  # simulated seconds; a fall can end the episode earlier
SEED = 0
RUN_HUMAN = False  # native viewer requires a desktop/OpenGL session


In [ ]:
def run_demo(render_mode, record_video=False, realtime=False):
    env = BounceEnv(
        xml_path=str(xml_path),
        reward=reward_objective,
        render_mode=render_mode,
        record_video=record_video,
        **env_kwargs,
    )
    frames = []
    terminated = truncated = False
    total_reward = 0.0
    try:
        obs, info = env.reset(seed=SEED)
        env.action_space.seed(SEED)
        frames.extend(env.pop_render_frames())
        started = time.perf_counter()
        for _ in range(env_kwargs["max_episode_steps"]):
            obs, reward, terminated, truncated, info = env.step(env.action_space.sample())
            total_reward += float(reward)
            # Camera frames are unstacked even if the observation has a history.
            frames.extend(env.pop_render_frames())
            if info["bounce_events"]:
                display(JSON({"rebonds": info["bounce_events"]}))
            if realtime:
                time.sleep(max(0., env.data.time - (time.perf_counter() - started)))
            if terminated or truncated or env.data.time >= SIM_DURATION:
                break
        metrics = dict(info["task_metrics"])
        metrics["evaluation_censored"] = not terminated
        return frames, metrics, total_reward
    finally:
        env.close()


def show_video(frames, fps):
    if not frames:
        print("Aucune image capturée.")
        return
    fig, ax = plt.subplots(figsize=(6, 4))
    ax.axis("off")
    image = ax.imshow(frames[0])
    def update(index):
        image.set_data(frames[index])
        return [image]
    clip = animation.FuncAnimation(fig, update, frames=len(frames),
                                   interval=1000 / fps, blit=True, repeat=False)
    plt.close(fig)
    display(HTML(clip.to_jshtml()))


## RGB et capture indépendante

`record_video=True` conserve toutes les images caméra, même en mode `rgb_array`.
Les événements sont détectés à chaque sous-pas physique. La chute termine immédiatement
l'épisode ; les mesures utilisent le temps simulé. La dispersion reste `null` avant deux impacts.


In [ ]:
frames_rgb, metrics_rgb, reward_rgb = run_demo("rgb_array", record_video=True)
display(JSON({"recompense_totale": reward_rgb, "metriques": metrics_rgb}))
show_video(frames_rgb, env_kwargs["camera_fps"])


## Liste d'images caméra

Le buffer est vidé à chaque pas pour conserver les images, y compris celles du pas terminal,
sans accumuler des images déjà consommées. L'animation utilise le FPS réel de configuration.


In [ ]:
frames_list, metrics_list, reward_list = run_demo("rgb_array_list")
display(JSON({"recompense_totale": reward_list, "metriques": metrics_list}))
show_video(frames_list, env_kwargs["camera_fps"])


## Viewer natif facultatif

Passer `RUN_HUMAN=True` dans la configuration sur une session de bureau compatible OpenGL.
La capture fonctionne aussi avec le viewer. La boucle reste bornée par la durée simulée,
la limite de pas et les conditions de fin d'épisode.


In [ ]:
if RUN_HUMAN:
    frames_human, metrics_human, reward_human = run_demo("human", record_video=True, realtime=True)
    display(JSON({"recompense_totale": reward_human, "metriques": metrics_human}))
    show_video(frames_human, env_kwargs["camera_fps"])
